# DB-QSP
https://arxiv.org/pdf/2504.01077

In [1]:
from qrisp import *
from qrisp.operators import X, Y, Z
from qrisp.jasp import q_fori_loop, q_cond, check_for_tracing_mode
from jax import lax
import scipy as sp
import numpy as np
import jax.numpy as jnp
from copy import deepcopy

## Example model: XXZ

In [2]:
from qrisp.vqe.problems.heisenberg import create_heisenberg_init_function, heisenberg_problem, create_heisenberg_hamiltonian
L = 5
G = nx.Graph()
G.add_edges_from([(k,(k+1)%L) for k in range(L-1)]) 
J = 1
B = 0.5
H = create_heisenberg_hamiltonian(G, J, B)
print(H)

X(0)*X(1) + X(1)*X(2) + X(2)*X(3) + X(3)*X(4) + Y(0)*Y(1) + Y(1)*Y(2) + Y(2)*Y(3) + Y(3)*Y(4) + 0.5*Z(0) + Z(0)*Z(1) + 0.5*Z(1) + Z(1)*Z(2) + 0.5*Z(2) + Z(2)*Z(3) + 0.5*Z(3) + Z(3)*Z(4) + 0.5*Z(4)


In [3]:
# Define scaling factor
F = 1

def exp_H(qv, t):
    H.trotterization(method='commuting')(qv,t/F,5)

# Hamiltonian simulation via second order Suzuki-Trotter formula with 2 steps
def exp_H_2(qv, t):
    H.trotterization(order=2,method='commuting')(qv,t/F,2)

# Calculate E and V

In [4]:
# in qrisp
def calculate_EV(H, state_prep):
    H_2 = H**2
    E = H.expectation_value(state_prep, diagonalisation_method="commuting")()
    E_2 = H_2.expectation_value(state_prep, diagonalisation_method="commuting")()
    
    V = E_2 - E**2
    
    return E, V

# matrix, for tests only
def compute_moments(psi, H):
    psi = np.array([psi]).transpose()
    E = (psi.conj().T @ H @ psi)[0,0].real
    S = (psi.conj().T @ H @ H @ psi)[0,0].real
    return E, S, S - E**2

## calculate s and phase
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$

with $s_k = \frac{-1}{\sqrt{V_k}}\arccos\left(\frac{|E_{k}-z_{k}|}{\sqrt{V_{k}+|E_{k}-z_{k}|^2}}\right)$
and $\theta_k = \arg\left(\frac{E_k-z_k}{|E_k-z_k|}\right).$

In [5]:
def QSP_unitary_synthesis_params(E, V, z):
    diff = E - z
    s = -1/jnp.sqrt(V)*jnp.arccos(jnp.abs(diff)/jnp.sqrt(V+jnp.abs(diff)**2))
    theta = jnp.angle(diff)
    return s, theta

## DB-QSP steps
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$
$$
e^{s_\Psi[\Psi,H]} = \left(
e^{is_\Psi^{(N)} \Psi}e^{is_\Psi^{(N)} H}
e^{-is_\Psi^{(N)} \Psi}e^{-is_\Psi^{(N)} H}
\right)^N \nonumber+O(s_\Psi^{3/2}/\sqrt N)\ , 
$$


### Numerical checks of DB-QSP

In [6]:
# Group commutator formula
psi = np.zeros(2**L)
psi[2**0+2**2+2**4] = 1
psi = psi/np.linalg.norm(psi)
psi_dm = np.outer(psi, psi.conj())
H_matrix = H.to_array()
z = -0.2
s, theta = QSP_unitary_synthesis_params(*compute_moments(psi, H_matrix)[::2], z)
print("s:", s)

comm = psi_dm@H_matrix - H_matrix@psi_dm
U_exact = sp.linalg.expm(s*comm)
# build the GC approx once
N = 5
for i in range(1, N+1):
    a = np.sqrt(abs(s/i))
    P, Hm = psi_dm, H_matrix
    A, B = 1j*a*P, 1j*a*Hm
    U_gc = np.eye(2**L, 2**L)
    for _ in range(i):
        U_gc = sp.linalg.expm(A) @ sp.linalg.expm(B) @ sp.linalg.expm(-A) @ sp.linalg.expm(-B) @ U_gc
    # Compare them:
    print(f"N={i} ‖U_exact - U_gc‖₂ =", np.linalg.norm(U_exact - U_gc))

s: -0.18731732822116817
N=1 ‖U_exact - U_gc‖₂ = 1.1387043704497093
N=2 ‖U_exact - U_gc‖₂ = 1.0375185417895263
N=3 ‖U_exact - U_gc‖₂ = 0.9233331809895404
N=4 ‖U_exact - U_gc‖₂ = 0.8347952556192932
N=5 ‖U_exact - U_gc‖₂ = 0.7661742142962871


In [7]:
def qsp_expect_1s(psi, H_matrix, z):
    L = int(np.log2(len(psi)))
    I = np.eye(2**L, 2**L)
    psi_target = (H_matrix-z*I)@ psi
    psi_target /= np.linalg.norm(psi_target)
    E = np.vdot(psi_target, H_matrix @ psi_target).real
    V = np.vdot(psi_target, H_matrix @ H_matrix @ psi_target).real - E**2
    return E, V, psi_target

def qsp_expect(psi, H_matrix, z):
    # z is list
    E, _, V = compute_moments(psi, H_matrix)
    E_ls = [E]
    V_ls = [V]
    psi_ls = []
    for zk in z:
        E, V, psi = qsp_expect_1s(psi, H_matrix, zk)
        E_ls.append(E)
        V_ls.append(V)
        psi_ls.append(psi)
    return E_ls, V_ls, psi_ls

def qsp_gc_expect_1s(psi, H_matrix, z, N=5):
    psi = np.array([psi]).flatten()
    psi_dm = np.outer(psi, psi.conj())
    s, theta = QSP_unitary_synthesis_params(*compute_moments(psi, H_matrix)[::2], z)
    s_ = np.sqrt(np.abs(s/N))
    U_qsp_gc = np.eye(H_matrix.shape[0], dtype=complex)
    for _ in range(N):
        U_qsp_gc = (
            sp.linalg.expm(1j*s_*psi_dm)
            @ sp.linalg.expm(1j*s_*H_matrix)
            @ sp.linalg.expm(-1j*s_*psi_dm)
            @ sp.linalg.expm(-1j*s_*H_matrix)
            @ U_qsp_gc
        )
    psi_final = sp.linalg.expm(1j*theta*psi_dm) @ U_qsp_gc @ psi
    E = np.vdot(psi_final, H_matrix @ psi_final).real
    V = np.vdot(psi_final, H_matrix @ H_matrix @ psi_final).real - E**2
    return E, V, psi_final

def qsp_gc_expect(psi, H_matrix, z, N=5):
    # z is list
    E, _, V = compute_moments(psi, H_matrix)
    E_ls = [E]
    V_ls = [V]
    psi_ls = []
    for zk in z:
        E, V, psi = qsp_gc_expect_1s(psi, H_matrix, zk, N)
        E_ls.append(E)
        V_ls.append(V)
        psi_ls.append(psi)
    return E_ls, V_ls, psi_ls

In [8]:
# example numpy calculation

zk = -0.2

# target state
E_target, V_target, psi_target = qsp_expect_1s(psi, H_matrix, zk)

# db-qsp state
s, theta = QSP_unitary_synthesis_params(*compute_moments(psi, H_matrix)[::2], zk)
print(f"s={s}, theta={theta}")
commutator = psi_dm@H_matrix - H_matrix@psi_dm
psi_qsp = sp.linalg.expm(1j*theta*psi_dm) @ sp.linalg.expm(s*commutator) @ psi
E_qsp_gc, V_qsp_gc, psi_qsp_gc = qsp_gc_expect_1s(psi, H_matrix, zk, N=3)
print("     Fidelity", abs(np.vdot(psi_target, psi_qsp))**2)
print("     Fidelity_GC", abs(np.vdot(psi_target, psi_qsp_gc))**2)
E_qsp = np.vdot(psi_qsp, H_matrix @ psi_qsp).real
V_qsp = np.vdot(psi_qsp, H_matrix @ H_matrix @ psi_qsp).real - E_qsp**2
print("After DB-QSP", (E_qsp, V_qsp))
print("After DB-QSP_GC", (E_qsp_gc, V_qsp_gc))

s=-0.18731732822116817, theta=3.141592653589793
     Fidelity 1.0
     Fidelity_GC 0.7946729091837044
After DB-QSP (np.float64(-7.0978544505653804), np.float64(7.488322447936845))
After DB-QSP_GC (np.float64(-6.437209809007314), np.float64(7.199343646251577))


### 1 step

In [9]:
# Apply k steps of DB-QSP (recursively)
def DB_QSP(qarg, U0, exp_H, N, s, theta, k=1):
    # s, theta are lists
    
    if k == 0:
        # |Psi_0> = U0|0>
        U0(qarg)
        
    else:

        def conjugator(qarg):
            with invert():
                DB_QSP(qarg, U0, exp_H, N, s, theta, k-1)
            
        def reflection(qarg, t_):
            with conjugate(conjugator)(qarg):
                if isinstance(qarg,QuantumArray):
                    qubits = sum([qv.reg for qv in qarg.flatten()], [])
                    mcp(t_, qubits, ctrl_state=0, method="khattar")
                else:
                    mcp(t_, qarg, ctrl_state=0, method="khattar")

        DB_QSP(qarg, U0, exp_H, N, s, theta, k-1)

        s_ = jnp.sqrt(jnp.abs(s[k-1])/N)
        theta_ = theta[k-1]

        for _ in range(N):
            exp_H(qarg, s_)
            reflection(qarg, -s_)
            exp_H(qarg, -s_)
            reflection(qarg, s_)

        reflection(qarg, theta_)

        return qarg

In [10]:
def U0(qv):
    x(qv[0])
    x(qv[2])
    x(qv[4])
    
def state_prep():
    qarg = QuantumVariable(L)
    U0(qarg) # Prepares the initial state |psi>
    return qarg

z = -0.2
N = 3
E, V = calculate_EV(H, state_prep)
s, theta = QSP_unitary_synthesis_params(E, V, z)
print("s, theta", (s, theta))
print("Initial EV:", E, V)

E_target, V_target, psi_target = qsp_expect_1s(psi, H_matrix, z)
E_qsp_gc, V_qsp_gc, psi_qsp_gc = qsp_gc_expect_1s(psi, H_matrix, z, N)
print("Exact value:", E_target, V_target)
print("DB-QSP expect:", E_qsp_gc, V_qsp_gc)
E1, V1 = calculate_EV(H, lambda: DB_QSP(QuantumVariable(L), U0, exp_H_2, N, [s], [theta]))
print("DB-QSP circuit:", E1, V1)

s, theta (Array(-0.18727816, dtype=float64, weak_type=True), Array(3.14159265, dtype=float64))
Initial EV: -4.499353051207403 16.029599884691823
Exact value: -7.097854450565385 7.488322447936824
DB-QSP expect: -6.437209809007314 7.199343646251577
                                                                                     

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


DB-QSP circuit: -6.446164315474115 7.066998995273693                                 


## multiple steps

$$
\frac{p(H)\ket{\Psi_0}}{\|p(H)\ket{\Psi_0}\|}=\prod_{k=0}^{K-1} e^{i \theta_k \Psi_{k}}  e^{s_{k}[\Psi_{k},H]}\ket{\Psi_0},
$$

In [11]:
# Apply multiple steps of DB-QSP 
from time import time
def DB_QSP_outer(U0, H, exp_H, z_list, N=2):
    start_time = time()
    L = H.find_minimal_qubit_amount()
    s_list = []
    theta_list = []

    def state_prep():
        qv = QuantumVariable(L)
        U0(qv)
        return qv

    E, V = calculate_EV(H, state_prep)
    print("Initial EV", E, V)
    E_list = [E]
    V_list = [V]
    
    for k, zk in enumerate(z_list):
        s, theta = QSP_unitary_synthesis_params(E, V, zk)
        s_list.append(s)
        theta_list.append(theta)

        E, V = calculate_EV(H, lambda: DB_QSP(QuantumVariable(L), U0, exp_H, N, s_list, theta_list, k+1))
        print(k, E, V, time() - start_time)
        E_list.append(E)
        V_list.append(V)


    return {
    "s": s_list,
    "theta": theta_list,
    "E": E_list,
    "V": V_list
    }


In [12]:
# Example usage of DB_QSP_outer
N = 2
z_list = [-0.2, 0.1j+0.1, 0.02]
E_qsp, V_qsp, psi_qsp = qsp_expect(psi, H_matrix, z_list)
print("E_qsp:", E_qsp)
E_qsp_gc, V_qsp_gc, psi_qsp_gc = qsp_gc_expect(psi, H_matrix, z_list, N=N)
print("E_qsp_gc:", E_qsp_gc)

E_qsp: [np.float64(-4.5), np.float64(-7.097854450565385), np.float64(-7.9747659670558), np.float64(-8.164036702757508)]
E_qsp_gc: [np.float64(-4.5), np.float64(-5.872050919472937), np.float64(-6.999223299519269), np.float64(-7.575657806622227)]


In [13]:
results = DB_QSP_outer(U0, H, exp_H_2, z_list, N)
print(results["E"])

Initial EV -4.4928554350730625 16.078332359265225                                    
0 -5.9000319999839395 10.826768513578884 3.950125217437744                           
1 -7.009314742338866 6.868376979897384 33.29621624946594                             
2 -7.5901688980201225 4.782474647487248 584.6594123840332                            
[-4.4928554350730625, -5.9000319999839395, -7.009314742338866, -7.5901688980201225]
